# Seleksi fitur: null importance, Boruta, permutasi, simulated annealing, algoritma genetik
**Kapan dipakai:** tabel teks + tabular dengan banyak fitur buatan (statistik teks, SVD, encoding,
fitur agregat) dan curiga sebagian hanya derau. Juga bahan slide: "fitur mana yang penting dan
kenapa kami membuang sisanya".

| cara | ide | sumber |
|---|---|---|
| null importance | gain asli vs gain saat target diacak (20×) | Kaggle, Olivier Grellier 2018 |
| Boruta | fitur harus mengalahkan "bayangan" acaknya berulang kali | Kursa & Rudnicki 2010 |
| permutasi | skor turun berapa saat kolom diacak | Breiman 2001 |
| annealing | balik 1–2 fitur per langkah, kadang terima yang lebih buruk | Kirkpatrick 1983 |
| genetik | populasi himpunan fitur, diawali hasil cara lain | Holland 1975 |

**Jujur:** semua pencarian hanya memakai 75% train ("bagian cari"). 25% sisanya (holdout) tidak
pernah dilihat dan menjadi juri akhir. Skor CV di bagian cari selalu optimis untuk
annealing/genetik.

In [ ]:
import os
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/seleksi_fitur_metaheuristik"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
TARGET = "recommended_ind"                   # GANTI saat lomba (biner, `rating` bocor)
NUM_COLS = ["age", "positive_feedback_count", "clothing_id"]               # GANTI saat lomba
CAT_COLS = ["division_name", "department_name", "class_name"]              # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]                                       # GANTI saat lomba
N_SVD = 20
HOLDOUT_SHARE = 0.25
N_FOLDS = 3
N_NULL = 20
NULL_PERCENTILE = 90
BORUTA_ROUNDS = 20
ANNEAL_STEPS = 50
ANNEAL_START = 0.002
ANNEAL_COOLING = 0.95
GA_POP = 10
GA_GENERATIONS = 6
SIZE_PENALTY = 0.0002        # hukuman per fitur saat annealing/genetik (pilih yang ramping)
MIN_GAIN = 0.001
SEED = 42
print("konfigurasi siap")

In [ ]:
def metric(truth, proba):
    return roc_auc_score(truth, proba)       # GANTI: metrik lomba (proba = peluang kelas 1)


def make_model():
    return lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=15, max_bin=63,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=SEED, verbose=-1)


print("uji metrik:", metric(np.array([0, 1, 1]), np.array([0.2, 0.7, 0.9])))

## Baca data + fitur (tabular, statistik teks, SVD TF-IDF)

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
both = pd.concat([train, test], ignore_index=True)
features = pd.DataFrame(index=both.index)
for col in NUM_COLS:
    features[col] = both[col]
for col in CAT_COLS:
    features[col] = both[col].astype("category").cat.codes
for col in TEXT_COLS:
    text = both[col].fillna("").astype(str)
    features[f"{col}_huruf"] = text.str.len()
    features[f"{col}_kata"] = text.str.split().str.len()
    features[f"{col}_seru"] = text.str.count("!")
    features[f"{col}_tanya"] = text.str.count(r"\?")
    features[f"{col}_kapital"] = text.str.count(r"[A-Z]") / (text.str.len() + 1)
    features[f"{col}_angka"] = text.str.count(r"\d")
full_text = both[TEXT_COLS].fillna("").agg(" . ".join, axis=1)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000,
                        sublinear_tf=True).fit_transform(full_text)
components = TruncatedSVD(N_SVD, random_state=SEED).fit_transform(tfidf)
for number in range(N_SVD):
    features[f"svd_{number}"] = components[:, number]
print("fitur:", features.shape)

## Tambah 10 kolom acak murni (HAPUS saat lomba)
Cara seleksi yang baik harus membuang semuanya; kolom "acak ikut" di tabel akhir mengukurnya.

In [ ]:
rng = np.random.default_rng(SEED)
for number in range(10):
    features[f"acak_{number}"] = rng.normal(size=len(features))
print("fitur + acak:", features.shape)

## Bagian cari (75%) + holdout (25%)

In [ ]:
ALL_COLUMNS = list(features.columns)
train_features = features.iloc[: len(train)].reset_index(drop=True)
labels_all = train[TARGET].to_numpy()
search_rows, holdout_rows = train_test_split(np.arange(len(train)), test_size=HOLDOUT_SHARE,
                                             stratify=labels_all, random_state=SEED)
search_part = train_features.iloc[search_rows]
holdout_part = train_features.iloc[holdout_rows]
y_search = labels_all[search_rows]
y_holdout = labels_all[holdout_rows]
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(search_part, y_search))
print("cari:", search_part.shape, "| holdout:", holdout_part.shape)

## Alat bersama: skor CV (dengan cache) + skor holdout

In [ ]:
CACHE = {}


def cv_score(columns):
    key = tuple(sorted(columns))
    if key not in CACHE:
        values = search_part[list(key)].to_numpy()
        oof = np.zeros(len(values))
        for idx_fit, idx_val in folds:
            model = make_model().fit(values[idx_fit], y_search[idx_fit])
            oof[idx_val] = model.predict_proba(values[idx_val])[:, 1]
        CACHE[key] = metric(y_search, oof)
    return CACHE[key]


def holdout_score(columns):
    model = make_model().fit(search_part[columns].to_numpy(), y_search)
    return metric(y_holdout, model.predict_proba(holdout_part[columns].to_numpy())[:, 1])


def gain_importance(values, labels):
    model = make_model().fit(values, labels)
    return model.booster_.feature_importance(importance_type="gain")


SELECTED = {"semua": ALL_COLUMNS}
SECONDS = {"semua": 0}
print("CV semua fitur:", round(cv_score(ALL_COLUMNS), 4))

## 1. Null importance
Fitur dipertahankan bila gain aslinya > persentil `NULL_PERCENTILE` gain saat target diacak.

In [ ]:
start = time.time()
values = search_part[ALL_COLUMNS].to_numpy()
actual = gain_importance(values, y_search)
null = np.zeros((N_NULL, len(ALL_COLUMNS)))
for number in range(N_NULL):
    null[number] = gain_importance(values, rng.permutation(y_search))
null_cut = np.percentile(null, NULL_PERCENTILE, axis=0)
SELECTED["null importance"] = list(np.array(ALL_COLUMNS)[actual > null_cut])
SECONDS["null importance"] = round(time.time() - start)
null_table = pd.DataFrame({"fitur": ALL_COLUMNS, "gain asli": actual.round(1),
                           "batas acak": null_cut.round(1)})
null_table.to_csv(f"{OUTPUT_FOLDER}/null_importance.csv", index=False)
print("dipertahankan:", len(SELECTED["null importance"]), "dari", len(ALL_COLUMNS))

## 2. Boruta (manual, tanpa paket)
Tiap putaran: semua kolom disalin lalu diacak per kolom ("bayangan"). "Menang" = gain kolom asli
> gain bayangan terbesar. Diterima bila menang jauh lebih sering dari 50% (uji binomial).

In [ ]:
def boruta_hits(values, labels):
    hits = np.zeros(values.shape[1])
    for _ in range(BORUTA_ROUNDS):
        shadow = rng.permuted(values, axis=0)
        importance = gain_importance(np.hstack([values, shadow]), labels)
        hits += importance[: values.shape[1]] > importance[values.shape[1]:].max()
    return hits


start = time.time()
hits = boruta_hits(values, y_search)
accepted = np.zeros(len(ALL_COLUMNS), dtype=bool)
for number in range(len(ALL_COLUMNS)):
    test_result = binomtest(int(hits[number]), BORUTA_ROUNDS, 0.5, alternative="greater")
    accepted[number] = test_result.pvalue < 0.05
SELECTED["Boruta"] = list(np.array(ALL_COLUMNS)[accepted])
SECONDS["Boruta"] = round(time.time() - start)
print("Boruta diterima:", accepted.sum(), "| menang per kolom:", hits.astype(int))

## 3. Permutation importance (lipatan pertama bagian cari)
Dipertahankan bila skor turun lebih dari 1 simpangan baku saat kolom diacak.

In [ ]:
start = time.time()
idx_fit, idx_val = folds[0]
model = make_model().fit(values[idx_fit], y_search[idx_fit])
found = permutation_importance(model, values[idx_val], y_search[idx_val], scoring="roc_auc",
                               n_repeats=5, random_state=SEED)       # GANTI scoring = metrik
keep = found.importances_mean - found.importances_std > 0
SELECTED["permutasi"] = list(np.array(ALL_COLUMNS)[keep])
SECONDS["permutasi"] = round(time.time() - start)
print("permutasi dipertahankan:", keep.sum())

## 4. Simulated annealing atas himpunan fitur
Skor = CV − `SIZE_PENALTY` × jumlah fitur. Mulai dari hasil null importance.

In [ ]:
def to_mask(columns):
    return np.isin(ALL_COLUMNS, columns)


def penalized(mask):
    return cv_score(list(np.array(ALL_COLUMNS)[mask])) - SIZE_PENALTY * mask.sum()


def flip_some(mask):
    trial = mask.copy()
    picked = rng.integers(len(mask), size=rng.integers(1, 3))
    trial[picked] = ~trial[picked]
    trial[rng.integers(len(mask))] |= not trial.any()
    return trial


print("uji flip:", flip_some(np.zeros(5, dtype=bool)).sum() >= 1)

In [ ]:
def annealing_mask(start_mask):
    current, current_value = start_mask.copy(), penalized(start_mask)
    best, best_value = current, current_value
    temperature = ANNEAL_START
    for _ in range(ANNEAL_STEPS):
        trial = flip_some(current)
        value = penalized(trial)
        if value > current_value or rng.random() < np.exp((value - current_value) / temperature):
            current, current_value = trial, value
        if current_value > best_value:
            best, best_value = current, current_value
        temperature *= ANNEAL_COOLING
    return best


start = time.time()
best_mask = annealing_mask(to_mask(SELECTED["null importance"]))
SELECTED["annealing"] = list(np.array(ALL_COLUMNS)[best_mask])
SECONDS["annealing"] = round(time.time() - start)
print("annealing:", len(SELECTED["annealing"]), "fitur")

## 5. Algoritma genetik (populasi awal = hasil cara 1–4 + acak)
Persilangan seragam, mutasi balik bit dengan peluang 1/jumlah fitur, 2 terbaik selalu lolos.

In [ ]:
def make_child(parents):
    first, second = rng.choice(len(parents), size=2, replace=False)
    child = np.where(rng.random(len(ALL_COLUMNS)) < 0.5, parents[first], parents[second])
    child = child ^ (rng.random(len(ALL_COLUMNS)) < 1 / len(ALL_COLUMNS))
    child[rng.integers(len(child))] |= not child.any()
    return child


def genetic_mask(seed_masks):
    population = list(seed_masks)
    while len(population) < GA_POP:
        population.append(rng.random(len(ALL_COLUMNS)) < 0.6)
    for _ in range(GA_GENERATIONS):
        values_now = np.zeros(len(population))
        for number in range(len(population)):
            values_now[number] = penalized(population[number])
        order = np.argsort(values_now)[::-1]
        parents = []
        for number in order[: GA_POP // 2]:
            parents.append(population[number])
        population = parents[:2]
        while len(population) < GA_POP:
            population.append(make_child(parents))
    return population[0]


print("fungsi genetik siap")

In [ ]:
start = time.time()
seed_masks = []
for name in ["semua", "null importance", "Boruta", "permutasi", "annealing"]:
    seed_masks.append(to_mask(SELECTED[name]))
SELECTED["genetik"] = list(np.array(ALL_COLUMNS)[genetic_mask(seed_masks)])
SECONDS["genetik"] = round(time.time() - start)
print("genetik:", len(SELECTED["genetik"]), "fitur | evaluasi tersimpan:", len(CACHE))

## Juri akhir: holdout yang tidak pernah dilihat

In [ ]:
rows = []
for name, columns in SELECTED.items():
    noise_kept = 0
    for column in columns:
        noise_kept += column.startswith("acak_")
    rows.append({"cara": name, "jumlah fitur": len(columns),
                 "CV cari": round(cv_score(columns), 4),
                 "holdout": round(holdout_score(columns), 4), "acak ikut": noise_kept,
                 "detik": SECONDS[name]})
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_seleksi.csv", index=False)
print(table.to_string(index=False))
best_row = table.iloc[int(table["holdout"].idxmax())]
CHOSEN = "semua"
if best_row["holdout"] >= table["holdout"][0] + MIN_GAIN:
    CHOSEN = best_row["cara"]
print("dipilih:", CHOSEN, "(holdout satu potong bisa goyang ±0,003: lihat juga CV)")

## Simpan fitur terpilih + gambar (bahan slide)

In [ ]:
votes = pd.DataFrame(index=ALL_COLUMNS)
for name, columns in SELECTED.items():
    votes[name] = np.isin(ALL_COLUMNS, columns).astype(int)
votes["suara"] = votes.drop(columns="semua").sum(axis=1)
votes.to_csv(f"{OUTPUT_FOLDER}/fitur_terpilih.csv")
pd.Series(SELECTED[CHOSEN]).to_csv(f"{OUTPUT_FOLDER}/fitur_{CHOSEN}.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].barh(table["cara"], table["holdout"], color="#0072B2")
axes[0].set_xlim(table["holdout"].min() - 0.005, table["holdout"].max() + 0.003)
axes[0].set_xlabel("skor holdout")
ordered = votes["suara"].sort_values()
axes[1].barh(ordered.index, ordered.to_numpy(), color="#E69F00")
axes[1].set_xlabel("jumlah cara yang mempertahankan fitur")
axes[1].tick_params(axis="y", labelsize=6)
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/seleksi_fitur.png", dpi=200, bbox_inches="tight")
plt.show()